# 05 — Analyse données complète (qualité, dates, patho)

Synthèse alignée sur la page web `/cisia/analyse` :

- **Cohorte** (décès exclus de l’entraînement)
- **Nulls & valeurs sentinelles** par CSV / colonne
- **Cohérence temporelle** (séjours, actes, bio, constantes, médicaments…)
- **Pathologies** associées au plus fort taux de retour 30 j

> Données synthétiques — certaines incohérences de dates sont **volontaires** (audit pédagogique).

In [ ]:
from pathlib import Path
import runpy

from IPython.display import Markdown, display

_sp = next(
    a / "setup_path.py"
    for a in [Path.cwd(), *Path.cwd().parents]
    if (a / "setup_path.py").is_file()
)
runpy.run_path(str(_sp))

%matplotlib inline
from _utils.bootstrap import paths
from src.data.analyse_donnees import build_analyse_complete

analyse = build_analyse_complete(paths)
print("Fichiers analysés :", len(analyse["resume_fichiers"]))


## 0. Synthèse graphique — exclus / hallucinations / incohérences

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

sq = analyse["stats_qualite"]
cats = sq["categories"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Volumes globaux
ax = axes[0]
labels = [c["label"] for c in cats]
vals = [c["n"] for c in cats]
colors = ["#515f74", "#ba1a1a", "#ba8c2e", "#2d6765"]
bars = ax.bar(range(len(vals)), vals, color=colors[: len(vals)])
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=20, ha="right")
ax.set_ylabel("Effectifs")
ax.set_title("Exclus / hallucinations / incohérences")
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width() / 2, v, str(v), ha="center", va="bottom", fontsize=9)

# Par fichier (mal notés + aberrants)
ax = axes[1]
par = [f for f in sq["par_fichier"] if f["n_mal_notes"] + f["n_aberrants"] > 0]
y = range(len(par))
ax.barh(y, [f["n_mal_notes"] for f in par], color="#ba1a1a", label="Mal notés")
ax.barh(
    y,
    [f["n_aberrants"] for f in par],
    left=[f["n_mal_notes"] for f in par],
    color="#ba8c2e",
    label="Aberrants",
)
ax.set_yticks(list(y))
ax.set_yticklabels([f["libelle"] for f in par])
ax.set_xlabel("Effectifs")
ax.set_title("Par fichier CSV")
ax.legend(loc="lower right")
fig.tight_layout()
display(fig)
plt.close(fig)

display(pd.DataFrame(cats))


## 1. Cohorte d’entraînement — exclusion des décès

In [ ]:
import pandas as pd

coh = analyse["cohorte"]
display(Markdown(f"**Règle :** {coh['regle']}"))
display(pd.DataFrame([
    {"indicateur": "Séjours bruts", "valeur": coh["n_sejours_bruts"]},
    {"indicateur": "Éligibles (Domicile)", "valeur": coh["n_eligibles_entrainement"]},
    {"indicateur": "Exclus", "valeur": coh["n_exclus"]},
    {"indicateur": "Dont décès exclus", "valeur": coh["n_deces_exclus"]},
]))
modes = pd.DataFrame([
    {"ModeSortie": k, "N": v, "Dans_entrainement": k == "Domicile"}
    for k, v in coh["repartition_mode_sortie"].items()
])
display(modes)


## 2. Nulls & valeurs sentinelles (résumé par fichier)

In [ ]:
resume = pd.DataFrame(analyse["resume_fichiers"])[
    ["fichier", "libelle", "n_lignes", "n_null_total", "n_sentinelle_total", "pct_null_moyen"]
]
display(Markdown(
    "Sentinelles numériques : "
    + ", ".join(str(x) for x in analyse["sentinelles_reference"]["numeriques"])
))
display(resume.sort_values("n_null_total", ascending=False))


## 3. Détail colonnes (null > 0 ou sentinelle > 0)

In [ ]:
detail = pd.DataFrame(analyse["nulls_sentinelles"])
detail_filtre = detail.loc[(detail["n_null"] > 0) | (detail["n_sentinelle"] > 0)].copy()
display(detail_filtre.sort_values(["n_null", "n_sentinelle"], ascending=False).head(40))
print(f"Colonnes impactées : {len(detail_filtre)} / {len(detail)}")


## 4. Cohérence temporelle

Contrôles :
- dates de séjour (sortie avant admission, durée incohérente)
- événements (actes, bio, constantes, médicaments, CR) hors fenêtre [admission, sortie]

In [ ]:
tempo = analyse["temporalite"]
display(Markdown("### Séjours"))
display(pd.DataFrame([tempo["sejours"]]).T.rename(columns={0: "valeur"}))

display(Markdown("### Événements vs fenêtre de séjour"))
evt = pd.DataFrame(tempo["evenements"])
cols = [
    c for c in [
        "fichier", "colonne_date", "n_lignes", "n_hors_fenetre_sejour",
        "pct_hors_fenetre_sejour", "n_avant_admission", "n_apres_sortie", "note",
    ]
    if c in evt.columns
]
display(evt[cols])


## 5. Pathologies chroniques et retour à 30 jours

Parmi les séjours curated portant chaque flag `patho_*`, quel taux de réadmission ?

In [ ]:
patho = pd.DataFrame(analyse["pathologies_readmission"])
if patho.empty:
    display(Markdown("_Parquet features absent — lancer `04_pipeline_donnees` ou `scripts/run_pipeline.py`."))
else:
    display(patho[["pathologie", "n_sejours", "pct_readmission", "pct_vs_global"]])
    top = patho.iloc[0]
    display(Markdown(
        f"**Plus associé au retour :** {top['pathologie']} "
        f"({top['pct_readmission']} % — écart +{top['pct_vs_global']} pts vs global)."
    ))


## 6. Lien webapp

La même synthèse est disponible pour le jury / soignant dans l’UI :

`http://127.0.0.1:8000/cisia/analyse`

## Suite du parcours

**Prochaine étape :** [01 — Benchmark & Optuna](../02_modeles/01_benchmark_optuna.ipynb)

| Étape | Notebook |
|-------|----------|
| → **Benchmark / Optuna** | [`../02_modeles/01_benchmark_optuna.ipynb`](../02_modeles/01_benchmark_optuna.ipynb) |
| Nettoyage / cohorte | [`../02_modeles/02_nettoyage_donnees.ipynb`](../02_modeles/02_nettoyage_donnees.ipynb) |
| Préparation features | [`../02_modeles/03_preparation_modele.ipynb`](../02_modeles/03_preparation_modele.ipynb) |
| Sommaire | [`../00_guide/00_sommaire.ipynb`](../00_guide/00_sommaire.ipynb) |
